# 05 · Learn recommendation weights from feedback
Fit an exploratory, regularized logistic regression on actual likes and dislikes. Text overlap is one signal. Profile–garment interaction features let preferences vary by observed face, body or skin value. An empty profile field supplies no signal. This experiment reads the local feedback database and prints coefficients.

In [ ]:
from pathlib import Path
import sys

# Start Jupyter in the project folder or its notebooks folder.
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "backend/catalogue.json").is_file()), None)
if root is None:
    raise RuntimeError("Open the project directory first. In Colab, upload/mount the project and %cd into it.")
sys.path.insert(0, str(root / "backend"))
import re, sqlite3, json
from collections import Counter
from contextlib import closing
import numpy as np

with closing(sqlite3.connect((root / "backend/outfit_feedback.sqlite3").as_uri() + "?mode=ro", uri=True)) as db:
    ratings = db.execute("SELECT clothing_request, outfit, face_shape, body_shape, skin_depth, rating FROM outfit_feedback WHERE rating IN ('like','dislike') ORDER BY id").fetchall()
catalogue = json.loads((root / "backend/catalogue.json").read_text())
tags = {row[0]: row[1] for row in catalogue}
print("Observed ratings:", dict(Counter(row[-1] for row in ratings)))

In [ ]:
def tokens(text):
    return set(re.findall(r"\w+", text.casefold()))

garments = set("shirt shirts jeans kurta kurtas jacket blazer chinos trousers dresses dress saree".split())
def features(row):
    request, outfit, face, body, skin, _ = row
    words = tokens(request)
    product = tokens(outfit + " " + tags.get(outfit, ""))
    values = {"text overlap": len(words & product) / max(1, len(words))}
    for field, value in [("face", face), ("body", body), ("skin", skin)]:
        if value and value.strip():
            for garment in product & garments:
                values[f"{field}={value.lower()} × {garment}"] = 1.0
    return values

def fit(rows):
    samples = [features(row) for row in rows]
    names = sorted({key for s in samples for key in s})
    x = np.array([[s.get(k, 0) for k in names] for s in samples], dtype=float)
    y = np.array([row[-1] == "like" for row in rows], dtype=float)
    w = np.zeros(len(names))
    bias = 0.0
    for _ in range(1200):
        error = 1 / (1 + np.exp(-np.clip(x @ w + bias, -40, 40))) - y
        w -= 0.1 * (x.T @ error / len(y) + 0.1 * w)
        bias -= 0.1 * error.mean()
    return dict(zip(names, w)), bias

if len(set(row[-1] for row in ratings)) == 2:
    weights, intercept = fit(ratings)
    for feature, weight in sorted(weights.items(), key=lambda pair: -abs(pair[1])):
        print(f"{weight:+.3f}  {feature}")
else:
    print("Collect both likes and dislikes before fitting.")

In [ ]:
# Hold out every rating for the same normalized query together.
groups = [" ".join(sorted(tokens(row[0]))) for row in ratings]
predictions = []
skipped = 0
for group in sorted(set(groups)):
    train = [row for row, g in zip(ratings, groups) if g != group]
    test = [row for row, g in zip(ratings, groups) if g == group]
    if len({r[-1] for r in train}) < 2:
        skipped += 1
        continue
    w, b = fit(train)
    for row in test:
        logit = b + sum(w.get(k, 0) * v for k, v in features(row).items())
        predictions.append((row[-1] == "like", logit >= 0))
print(f"Scored {len(predictions)}/{len(ratings)} held-out ratings. Skipped query groups: {skipped}.")
if not skipped and {p[0] for p in predictions} == {False, True}:
    recalls = [np.mean([pred == truth for truth, pred in predictions if truth == label]) for label in [False, True]]
    print(f"Balanced accuracy: {np.mean(recalls):.1%}")
else:
    print("Insufficient class coverage across requests to report balanced accuracy.")